# Story 1.2: Simulation Graph E2E Test

This notebook tests the LangGraph simulation workflow end-to-end with real AI agents.

**What this tests:**
1. Complete scenario generation
2. LangGraph workflow with interrupt() for human-in-the-loop
3. Full simulation cycle through multiple turns
4. MLflow tracing of multi-step execution

## 1. Setup & Imports

In [1]:
import warnings

# Suppress MLflow/LangGraph autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig
from summit_sim.settings import settings
from summit_sim.tracing import (
    enable_tracing,
    log_simulation_results,
    simulation_session,
)

# Initialize MLflow with unified tracing for Pydantic AI and LangGraph
enable_tracing()

print(f"✓ MLflow tracking URI: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment name: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key configured: {bool(settings.openrouter_api_key)}")
print("\n📊 MLflow Tracing Architecture:")
print("  • Scenario generation → Independent Pydantic AI traces")
print("  • Simulation loop → Nested under parent run with session_id")
print("  • class_id → Links generation and simulation traces in MLflow")
print("  • Interrupt 'exceptions' → Expected (human-in-the-loop mechanism)")

✓ MLflow tracking URI: https://mlflow.bhamm-lab.com
✓ Experiment name: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key configured: True

📊 MLflow Tracing Architecture:
  • Scenario generation → Independent Pydantic AI traces
  • Simulation loop → Nested under parent run with session_id
  • class_id → Links generation and simulation traces in MLflow
  • Interrupt 'exceptions' → Expected (human-in-the-loop mechanism)


## 2. Generate Scenario

First, we'll generate a complete scenario with multiple turns.

**Note**: Scenario generation creates an independent Pydantic AI trace in MLflow, separate from the simulation session. This is by design - generation and simulation are distinct phases.

In [2]:
# Test configuration - class_id auto-generated if not provided
config = HostConfig(num_participants=3, activity_type="hiking", difficulty="med")

print(f"Generating scenario: {config}")
print(f"📋 class_id: {config.class_id}")
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Starting turn: {scenario.starting_turn_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

print("\n💡 In MLflow UI, filter by class_id to see all related traces")

Generating scenario: num_participants=3 activity_type='hiking' difficulty='med' class_id='68a670'
📋 class_id: 68a670
------------------------------------------------------------
✓ Title: The Alpine Lag: Managing Heat Exhaustion
✓ Setting: Alpine hiking trail at 7,000ft elevation. Sunny afternoon, moderate physical exertion, 2 hours from the trailhead.
✓ Patient: Age: 32M. Appearing visibly fatigued, staggering on a moderate alpine trail. Skin is flushing red, cool and clammy to the touch, tachycardia (rapid pulse). Patient is thirsty but complained of nausea recently.
✓ Turns: 4
✓ Starting turn: 0

Learning Objectives:
  • Systematic assessment of a hiking patient experiencing physical distress.
  • Differentiating Heat Exhaustion from more severe environmental emergencies.
  • Proper field management of exertional heat illness.

💡 In MLflow UI, filter by class_id to see all related traces


Trace(trace_id=tr-5fc7e1a82563e06f341c8577ae23274d)

## 3. Display Scenario Turns

Let's see what turns were generated:

In [3]:
for _i, turn in enumerate(scenario.turns):
    print()
    print("=" * 60)
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"Turn {turn.turn_id} {marker}")
    print("=" * 60)
    print()
    print(turn.narrative_text)
    print()
    print("Choices:")
    for choice in turn.choices:
        if choice.next_turn_id is None:
            end = "END"
        else:
            end = f"Turn {choice.next_turn_id}"
        mark = "✓" if choice.is_correct else " "
        print(f"  [{mark}] {choice.description} -> {end}")


Turn 0 (START)

You are hiking with two friends on a moderate alpine trail. About 2 hours from the trailhead, one of your companions, a 32-year-old male, begins staggering. He complains of dizziness and nausea, and he is visibly flushed. He is lagging behind the group and his speech seems slightly disorganized.

Choices:
  [ ] Encourage the patient to push through the last 2 miles to reach the trailhead quickly. -> Turn 1
  [✓] Immediately stop the group, move the patient to shade, and begin a systematic assessment. -> Turn 2

Turn 1 

Urging the patient to keep going proves disastrous. The intensity of activity causes his condition to rapidly decline. He sits down, unable to continue. You must now stop and conduct an assessment.

Choices:
  [ ] The patient stops walking, collapses, and loses consciousness due to exertion. Scenario ends unexpectedly. -> END
  [✓] (Correction) Realize mistake, stop the group, and conduct assessment. -> Turn 2

Turn 2 

You have moved the patient to a s

## 4. Initialize Simulation Graph

Create the LangGraph workflow with checkpointer for interrupt support:

In [4]:
# Create the graph with in-memory checkpointer
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state (TypedDict - use dict syntax)
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "class_id": config.class_id,
}

print("✓ Graph created successfully")
print(f"✓ Initial state: turn_id={initial_state['current_turn_id']}")
print(f"✓ class_id: {initial_state['class_id']}")
print("\n⚠️  Next cell will start MLflow session with parent run tracking")

✓ Graph created successfully
✓ Initial state: turn_id=0
✓ class_id: 68a670

⚠️  Next cell will start MLflow session with parent run tracking


## 5. Run Simulation with MLflow Session Tracking

Execute the full simulation workflow within an MLflow session. The simulation will:
1. Create a parent MLflow run with session metadata
2. Present each turn (with traces nested under parent)
3. Pause with `interrupt()` for your choice
4. Call the AI agent for feedback
5. Advance to next turn
6. Log final results to the parent run

**MLflow Features:**
- Session ID links all traces together
- Parent run contains all agent calls as child spans
- Automatic error tracking (status: completed/failed)
- Final metrics: total_turns, is_complete, learning_moments_count

**⚠️ Expected Behavior:**
- `interrupt()` raises exceptions for human-in-the-loop - these appear in MLflow traces but are NOT errors
- Scenario generation (Cell 2) creates separate Pydantic AI traces
- Only simulation traces are nested under the parent run

In [5]:
from langgraph.types import Command

# Wrap simulation in session context for unified MLflow tracking
with simulation_session(config) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print(
        f"📊 Session Name: sim-{config.activity_type}-{config.num_participants}p-{config.difficulty}-...\n"
    )

    current_state = initial_state
    turn_count = 0
    simulation_complete = False

    print("\n=== STARTING SIMULATION ===\n")

    while not simulation_complete:
        turn_count += 1
        print(f"--- TURN {turn_count} ---")

        async for event in graph.astream(current_state, graph_config):
            if "__interrupt__" in event:
                interrupt_obj = event["__interrupt__"]
                if hasattr(interrupt_obj, "value"):
                    interrupt_data = interrupt_obj.value
                elif isinstance(interrupt_obj, (list, tuple)):
                    interrupt_data = (
                        interrupt_obj[0].value
                        if hasattr(interrupt_obj[0], "value")
                        else interrupt_obj[0]
                    )
                else:
                    interrupt_data = interrupt_obj

                print(f"\n📖 {interrupt_data['narrative']}\n")
                print("Choices:")
                for i, choice in enumerate(interrupt_data["choices"], 1):
                    print(f"  {i}. {choice['description']}")

                while True:
                    try:
                        sel = int(input("\nSelect: ")) - 1
                        if 0 <= sel < len(interrupt_data["choices"]):
                            break
                        print("Invalid")
                    except ValueError:
                        print("Enter number")

                selected = interrupt_data["choices"][sel]
                current_state = await graph.ainvoke(
                    Command(resume={"choice_id": selected["choice_id"]}), graph_config
                )
                break

            # Check for graph completion
            if event == {}:
                simulation_complete = True
                break

        # Check if state indicates completion (AppState is TypedDict, use dict access)
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("Safety stop")
            break

    print("\n=== SIMULATION COMPLETE ===")

    # Log final results to MLflow parent run
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
    )
    print("\n✓ Results logged to MLflow")

2026/03/23 16:20:56 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7c0eaff510> at 0x7f7bce0d34c0> was created in a different Context
2026/03/23 16:20:56 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7c0eaff510> at 0x7f7bce143940> was created in a different Context



📋 Session ID: e5d8f74b-a210-48f3-96a9-80d4606742e2
📊 Session Name: sim-hiking-3p-med-...


=== STARTING SIMULATION ===

--- TURN 1 ---

📖 You are hiking with two friends on a moderate alpine trail. About 2 hours from the trailhead, one of your companions, a 32-year-old male, begins staggering. He complains of dizziness and nausea, and he is visibly flushed. He is lagging behind the group and his speech seems slightly disorganized.

Choices:
  1. Encourage the patient to push through the last 2 miles to reach the trailhead quickly.
  2. Immediately stop the group, move the patient to shade, and begin a systematic assessment.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/23 16:20:57 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7c0eaff510> at 0x7f7bce0d3a00> was created in a different Context
2026/03/23 16:21:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7c0eaff510> at 0x7f7bce1853c0> was created in a different Context
2026/03/23 16:21:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7c0eaff510> at 0x7f7bce140440> was created in a different Context
2026/03/23 16:21:02 WARNING mlflow.utils.autologging_utils: Encountered un

--- TURN 2 ---

📖 Urging the patient to keep going proves disastrous. The intensity of activity causes his condition to rapidly decline. He sits down, unable to continue. You must now stop and conduct an assessment.

Choices:
  1. The patient stops walking, collapses, and loses consciousness due to exertion. Scenario ends unexpectedly.
  2. (Correction) Realize mistake, stop the group, and conduct assessment.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 16:21:03 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7c0eaff510> at 0x7f7bce03d340> was created in a different Context
2026/03/23 16:21:05 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa

...

--- TURN 4 ---

📖 The patient is resting in the shade. How does your treatment plan progress?

Choices:
  1. The patient vomits from the rapid fluid intake, becomes more confused, and requires an emergency evacuation. Proper treatment was delayed.
  2. The patient rests in the shade, sips fluids slowly, and makes a steady recovery over the next 45 minutes, allowing for a slow, assisted walk out of the backcountry.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 16:21:17 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7c0eaff510> at 0x7f7bcdee9b40> was created in a different Context
2026/03/23 16:21:25 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


=== SIMULATION COMPLETE ===

✓ Results logged to MLflow
🏃 View run sim-68a670-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/171f1e8512b9450ba0bc0655d4eeeec5
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-18b0dfe5469591055fb02bffd44647a4), Trace(trace_id=tr-dd21a397248e5fc3e646aa622244348a), Trace(trace_id=tr-0b1756b32b9a878a0cd6c8f83ebc49bf), Trace(trace_id=tr-53097f7b7a42d6e3a08e217159b9132b)]

2026/03/23 16:21:26 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f7c0eaff510> at 0x7f7bce0e6c00> was created in a different Context


## 6. Display Results

Show the complete transcript and learning moments:

In [6]:
print("\n" + "=" * 60)
print("TRANSCRIPT")
print("=" * 60 + "\n")

for _i, entry in enumerate(current_state["transcript"], 1):
    print(f"Turn {entry['turn_id']}:")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print(f"  Learning: {', '.join(entry['learning_moments'])}")
    print()


TRANSCRIPT

Turn 0:
  Situation: You are hiking with two friends on a moderate alpine trail. About 2 hours from t...
  Choice: Encourage the patient to push through the last 2 miles to reach the trailhead quickly.
  Feedback: Encouraging a symptomatic hiker to "push through" is a critical error, as it risks their condition r...
  Learning: Recognize that staggering and disorganized speech are serious indicators of neurological involvement in heat illness, requiring an immediate stop to all exertion., The mantra 'Stop, Cool, Hydrate' is essential; moving a heat-exhausted patient further increases internal heat production and significantly increases the likelihood of a medical emergency.

Turn 0:
  Situation: You are hiking with two friends on a moderate alpine trail. About 2 hours from t...
  Choice: Encourage the patient to push through the last 2 miles to reach the trailhead quickly.
  Feedback: Encouraging a symptomatic hiker to "push through" is a critical error, as it risks their 

In [7]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for _i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{_i}. {moment}")


KEY LEARNING MOMENTS

1. Recognize that staggering and disorganized speech are serious indicators of neurological involvement in heat illness, requiring an immediate stop to all exertion.
2. The mantra 'Stop, Cool, Hydrate' is essential; moving a heat-exhausted patient further increases internal heat production and significantly increases the likelihood of a medical emergency.
3. Recognize that staggering and disorganized speech are serious indicators of neurological involvement in heat illness, requiring an immediate stop to all exertion.
4. The mantra 'Stop, Cool, Hydrate' is essential; moving a heat-exhausted patient further increases internal heat production and significantly increases the likelihood of a medical emergency.
5. Recognize that staggering and disorganized speech are serious indicators of neurological involvement in heat illness, requiring an immediate stop to all exertion.
6. The mantra 'Stop, Cool, Hydrate' is essential; moving a heat-exhausted patient further incre

## 7. MLflow Verification

Check your MLflow UI to verify:

### Trace Architecture
- **Scenario Generation** (Cell 2): Independent Pydantic AI trace
  - Tagged with `class_id` for linking to simulation
  - Appears as standalone trace in MLflow

- **Simulation** (Cell 5): Parent run with nested LangGraph traces
  - Parent Run: `sim-{class_id}-{activity}-{participants}p-{difficulty}`
  - Tagged with same `class_id` as generation
  - Session ID: UUID used for thread_id grouping
  - Child spans: Each agent call nested under parent

### Linking Generation and Simulation
Both phases share the same `class_id` tag:
- Filter in MLflow UI: `tags.class_id = "{class_id}"`
- Or search: `{class_id}` in run names
- This links: generation trace → simulation parent run

### Expected Behaviors
- **Interrupt 'Exceptions'**: Normal! LangGraph's `interrupt()` raises exceptions
  for human-in-the-loop. These appear in traces but are not errors.
- **Evaluation Tab**: Runs with traces appear here in MLflow 3.x UI
- **Session View**: Filter by thread_id to see grouped traces

### Logged Data
- **Metrics**: total_turns, is_complete, learning_moments_count
- **Tags**: class_id, session_type, activity_type, difficulty, status
- **Params**: class_id, session_id (UUID), activity_type, num_participants, difficulty

In [8]:
import mlflow

# Show experiment info
experiment = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"\nView traces at: {settings.mlflow_tracking_uri}")
print("\nIn MLflow UI, look for:")
print(f"  📁 class_id: {config.class_id}")
print(f"  📊 Generation trace: Filter by class_id tag or search '{config.class_id}'")
print(f"  📊 Simulation run: sim-{config.class_id}-... (parent run with nested traces)")
print("\nKey Features:")
print(f"  • class_id '{config.class_id}' links generation and simulation")
print('  • Filter: tags.class_id = "<your-class-id>"')
print("  • Metrics logged at session completion")
print("  • Status tag shows completed/failed")

Experiment ID: 7
Artifact Location: mlflow-artifacts:/7

View traces at: https://mlflow.bhamm-lab.com

In MLflow UI, look for:
  📁 class_id: 68a670
  📊 Generation trace: Filter by class_id tag or search '68a670'
  📊 Simulation run: sim-68a670-... (parent run with nested traces)

Key Features:
  • class_id '68a670' links generation and simulation
  • Filter: tags.class_id = "<your-class-id>"
  • Metrics logged at session completion
  • Status tag shows completed/failed


---
## ✅ E2E Test Complete

If you successfully completed the simulation:
- ✓ LangGraph workflow executed correctly
- ✓ Human-in-the-loop interrupt worked
- ✓ AI feedback generated for each choice
- ✓ State advanced through all turns
- ✓ MLflow captured multi-step execution